# Attention Is All You Need - 실습 코드 1: 논문 핵심: Scaled Dot-Product Attention 구현

- Tutorial ID: `expand-attention-is-all-you-need`
- Tutorial: Attention Is All You Need
- Section ID: `expand-attention-is-all-you-need-code-1`
- Section: 실습 코드 1: 논문 핵심: Scaled Dot-Product Attention 구현


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: 논문 핵심: Scaled Dot-Product Attention 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    """Vaswani et al. 논문의 핵심 Attention 구현"""
        def __init__(self, d_k, dropout=0.1):
        super().__init__()
        self.scale = math.sqrt(d_k)
                self.dropout = nn.Dropout(dropout)

        def forward(self, Q, K, V, mask=None):
        # Q: (batch, heads, seq_q, d_k)
        # K: (batch, heads, seq_k, d_k)
        # V: (batch, heads, seq_v, d_v)
                scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        if mask is not None:
                        scores = scores.masked_fill(mask == 0, -1e9)

                attn_weights = self.dropout(F.softmax(scores, dim=-1))
                output = torch.matmul(attn_weights, V)
        return output, attn_weights


class MultiHeadAttention(nn.Module):
    """논문의 Multi-Head Attention 완전 구현"""
        def __init__(self, d_model=512, num_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        
        # 논문: W_Q, W_K, W_V, W_O 각각 d_model x d_model
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        
        self.attention = ScaledDotProductAttention(self.d_k, dropout)
                self.dropout = nn.Dropout(dropout)

        def forward(self, query, key, value, mask=None):
                batch_size = query.size(0)
        
        # 선형 변환 + multi-head 분할
                Q = self.W_Q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
                K = self.W_K(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
                V = self.W_V(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Scaled Dot-Product Attention
                attn_out, attn_weights = self.attention(Q, K, V, mask)
        
        # Concat heads + 최종 선형 변환
                attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
                output = self.W_O(attn_out)
        
        return output, attn_weights


# 논문의 Label Smoothing 구현
class LabelSmoothing(nn.Module):
    """논문 Section 5.4: label_smoothing = 0.1"""
        def __init__(self, vocab_size, padding_idx=0, smoothing=0.1):
        super().__init__()
        self.criterion = nn.KLDivLoss(reduction='sum')
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.vocab_size = vocab_size

        def forward(self, pred, target):
        true_dist = torch.zeros_like(pred)
        true_dist.fill_(self.smoothing / (self.vocab_size - 2))
        true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0
                mask = torch.nonzero(target == self.padding_idx)
        if mask.dim() > 0:
            true_dist.index_fill_(0, mask.squeeze(), 0.0)
        return self.criterion(pred, true_dist)


# 논문의 Learning Rate Schedule 구현
class TransformerLRScheduler:
    """논문 Section 5.3: warmup + decay 스케줄러"""
        def __init__(self, optimizer, d_model=512, warmup_steps=4000):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.steps = 0

        def step(self):
        self.steps += 1
        lr = self.d_model ** (-0.5) * min(
            self.steps ** (-0.5),
            self.steps * self.warmup_steps ** (-1.5)
        )
                for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        return lr


# 사용 예시
mha = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.randn(2, 10, 512)  # batch=2, seq=10, d=512
out, weights = mha(x, x, x)
print(f"Input: {x.shape} → Output: {out.shape}")
print(f"Attention weights: {weights.shape}")
print(f"Parameters: {sum(p.numel() for p in mha.parameters()):,}")